# 09 - Execução do Pipeline

## Descrição

Este notebook apresenta um guia prático, passo a passo, para executar o pipeline completo do projeto **Engenharia de Dados — Projeto Final** em ambiente local. O objetivo é permitir que qualquer pessoa (desenvolvedor, QA, arquiteto ou estudante) consiga levantar a infraestrutura, gerar os dados, criar as estruturas do Data Lake, executar as DAGs e validar os resultados — tudo em uma única máquina com Docker.

---

## Pré-requisitos

| Componente | Versão Mínima | Verificação |
|-----------|--------------|-------------|
| **Docker** | 20.10+ | `docker --version` |
| **Docker Compose** | 2.0+ (plugin) | `docker compose version` |
| **Python** | 3.11+ | `python3 --version` |
| **Git** | 2.30+ | `git --version` |

### Instalação (Ubuntu/Debian)

```bash
# Instalar Docker (se ainda não tiver)
sudo apt update
sudo apt install -y docker.io docker-compose-plugin

# Adicionar seu usuário ao grupo docker (evita usar sudo)
sudo usermod -aG docker $USER
newgrp docker

# Verificar
docker --version
docker compose version
```

---

## Passo 1: Clonar e Configurar o Repositório

### 1.1 Clonar o repositório

```bash
git clone https://github.com/olucasoliverio/Engenharia_Dados_Final.git
cd Engenharia_Dados_Final
```

### 1.2 Criar o arquivo `.env`

Copie o arquivo de exemplo e ajuste se necessário (valores padrão funcionam para ambiente local):

```bash
cp .env.example .env
```

O arquivo `.env` contém todas as variáveis necessárias. Para execução local, os valores padrão já estão corretos.

### 1.3 Criar ambiente virtual Python (opcional, para scripts)

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
```

---

## Passo 2: Subir a Infraestrutura Base

### 2.1 Levantar MongoDB e MinIO

```bash
docker compose up -d mongodb minio
```

### 2.2 Verificar saúde dos serviços

```bash
docker compose ps
```

Saída esperada:

```text
NAME                COMMAND                  SERVICE             STATUS              PORTS
mongodb_ecommerce   "docker-entrypoint.s…"   mongodb             running (healthy)   27017/tcp
minio_datalake      "/usr/bin/docker-e…"   minio               running (healthy)   0.0.0.0:9000-9001->9000-9001/tcp
```

> **Nota**: Aguarde o STATUS `healthy` antes de prosseguir. A primeira inicialização pode levar 30-60 segundos.

### 2.3 Acessar o console MinIO (opcional)

- URL: http://localhost:9001
- Usuário: `minioadmin`
- Senha: `minioadmin`

Você verá o bucket `datalake` (vazio, ainda).

---

## Passo 3: Gerar e Carregar Dados no MongoDB

### 3.1 Instalar dependências do grupo `dataset`

```bash
pip install ".[dataset]"
```

Isso instala: `pymongo`, `dnspython`, `faker`, `pandas`.

### 3.2 Gerar e carregar os dados

```bash
# Carrega o .env no ambiente atual
set -a && source .env && set +a

# Executa o script de carga (gera CSVs se necessário e carrega no MongoDB)
python dataset/scripts_py/carregar_mongo.py
```

Saída esperada:

```text
Gerando CSVs...
[clientes] 15000 registros em dataset/arquivos_csv/clientes.csv
[categorias] 15000 registros em dataset/arquivos_csv/categorias.csv
...
[avaliacoes] 15000 registros em dataset/arquivos_csv/avaliacoes.csv

Carregando MongoDB...
[clientes] Recriada + 15000 documentos (batch 5000)
[categorias] Recriada + 15000 documentos (batch 5000)
...
[avaliacoes] Recriada + 15000 documentos (batch 5000)

Carga concluida: 10 colecoes, 150000 documentos totais.
```

> **Nota**: O script é idempotente — se executar novamente, recriará todas as coleções.

### 3.3 Verificar os dados no MongoDB (opcional)

```bash
# Entrar no container do MongoDB
docker compose exec mongodb mongosh -u admin -p admin123 --authenticationDatabase admin ecommerce

# No shell mongosh, execute:
show collections
db.clientes.countDocuments()
db.pedidos.findOne()

# Sair
exit
```

---

## Passo 4: Criar Estruturas do Data Lake (MinIO/S3)

### 4.1 Instalar dependências do grupo `infra`

```bash
pip install ".[infra]"
```

Isso instala: `boto3`.

### 4.2 Criar Landing

```bash
set -a && source .env && set +a
python scripts/criar_estrutura_landing.py
```

Saída esperada:

```text
Estrutura criada/atualizada: 10 colecoes em s3://datalake/landing/
Estrutura Landing valida: s3://datalake/landing/
```

### 4.3 Criar Bronze

```bash
python scripts/criar_estrutura_bronze.py
```

Saída esperada:

```text
Estrutura criada/atualizada: 10 tabelas em s3://datalake/bronze/
Marcadores criados: 10; manifesto atualizado: sim
Estrutura Bronze valida: s3://datalake/bronze/
```

### 4.4 Criar Silver

```bash
python scripts/criar_estrutura_silver.py
```

### 4.5 Criar Gold

```bash
python scripts/criar_estrutura_gold.py
```

### 4.6 Verificar no console MinIO (opcional)

Acesse http://localhost:9001 e navegue no bucket `datalake`. Você verá:

```text
datalake/
├── landing/
│   ├── ecommerce/
│   │   ├── clientes/_READY
│   │   ├── categorias/_READY
│   │   └── ...
│   └── _control/_structure.json
├── bronze/
│   ├── ecommerce/
│   │   ├── clientes/_READY
│   │   └── ...
│   └── _control/_structure.json
├── silver/
│   └── ...
└── gold/
    └── ...
```

---

## Passo 5: Levantar o Apache Airflow

### 5.1 Subir o stack completo do Airflow

```bash
docker compose up -d --build \
  airflow-apiserver \
  airflow-scheduler \
  airflow-dag-processor \
  airflow-triggerer
```

> **Nota**: A primeira build da imagem customizada pode levar 5-10 minutos.

### 5.2 Verificar saúde do Airflow

```bash
docker compose ps
```

Saída esperada:

```text
NAME                   STATUS              PORTS
airflow_apiserver      running (healthy)   0.0.0.0:8080->8080/tcp
airflow_scheduler      running (healthy)   
airflow_dag_processor  running (healthy)   
airflow_triggerer      running (healthy)   
airflow_postgres       running (healthy)   5432/tcp
mongodb_ecommerce      running (healthy)   27017/tcp
minio_datalake         running (healthy)   0.0.0.0:9000-9001->9000-9001/tcp
```

### 5.3 Acessar a interface web do Airflow

- URL: http://localhost:8080
- Usuário: `airflow`
- Senha: `airflow`

### 5.4 Verificar importação das DAGs

```bash
docker compose exec airflow-apiserver airflow dags list
```

Saída esperada:

```text
dag_id              | filepath                    | owner             | paused
====================+=============================+===================+=======
mongodb_to_landing  | /opt/airflow/dags/...       | engenharia_dados  | True
landing_to_bronze   | /opt/airflow/dags/...       | engenharia_dados  | True
bronze_to_silver    | /opt/airflow/dags/...       | engenharia_dados  | True
silver_to_gold      | /opt/airflow/dags/...       | engenharia_dados  | True
```

Verificar erros de importação:

```bash
docker compose exec airflow-apiserver airflow dags list-import-errors
```

Saída esperada: vazia (nenhum erro).

### 5.5 Ativar as DAGs (opcional, para execução automática)

Por padrão, as DAGs são criadas pausadas (`AIRFLOW__CORE__DAGS_ARE_PAUSED_AT_CREATION: "true"`).

Para ativar execução automática (agendamento a cada 15 min):

```bash
docker compose exec airflow-apiserver airflow dags unpause mongodb_to_landing
docker compose exec airflow-apiserver airflow dags unpause landing_to_bronze
docker compose exec airflow-apiserver airflow dags unpause bronze_to_silver
docker compose exec airflow-apiserver airflow dags unpause silver_to_gold
```

Ou ative pela UI web (toggle ao lado de cada DAG).

---

## Passo 6: Executar as DAGs (Manualmente)

Para execução controlada e observação passo a passo, execute cada DAG manualmente na ordem.

### 6.1 Executar DAG 1: `mongodb_to_landing`

```bash
docker compose exec airflow-apiserver airflow dags trigger mongodb_to_landing
```

Acompanhe pela UI: http://localhost:8080/dags/mongodb_to_landing/grid

Verificar resultado no MinIO:

```bash
# Listar objetos na Landing
docker compose exec minio mc ls local/datalake/landing/ecommerce/
```

Ou pelo console MinIO: http://localhost:9001 → bucket `datalake` → prefixo `landing/ecommerce/clientes/`.

Você verá arquivos `part-00000.json` dentro de pastas `extraction_date=.../run_id=.../`.

### 6.2 Executar DAG 2: `landing_to_bronze`

```bash
docker compose exec airflow-apiserver airflow dags trigger landing_to_bronze
```

Acompanhe pela UI: http://localhost:8080/dags/landing_to_bronze/grid

Verificar resultado no MinIO:

- Console: `bronze/ecommerce/clientes/` — você verá `_delta_log/` e arquivos Parquet em `ingestion_date=.../`.

### 6.3 Executar DAG 3: `bronze_to_silver`

```bash
docker compose exec airflow-apiserver airflow dags trigger bronze_to_silver
```

Acompanhe pela UI: http://localhost:8080/dags/bronze_to_silver/grid

Verificar resultado no MinIO:

- Console: `silver/ecommerce/clientes/` — `_delta_log/` e arquivos Parquet.
- Console: `silver/_control/quality_log/` — registros rejeitados (se houver).

### 6.4 Executar DAG 4: `silver_to_gold`

```bash
docker compose exec airflow-apiserver airflow dags trigger silver_to_gold
```

Acompanhe pela UI: http://localhost:8080/dags/silver_to_gold/grid

Verificar resultado no MinIO:

- Console: `gold/ecommerce/dim_cliente/` — `_delta_log/` e Parquet (com colunas `dw_valid_from`, `dw_valid_to`, `dw_is_current`).
- Console: `gold/ecommerce/fato_vendas/` — `_delta_log/` e Parquet particionado por `ano=...`.

---

## Passo 7: Executar o Pipeline Completo (Automático)

Após ativar as DAGs (Passo 5.5), o pipeline executa automaticamente a cada 15 minutos:

| DAG | Agendamento | Offset |
|-----|-------------|--------|
| `mongodb_to_landing` | `*/15 * * * *` | 0 min |
| `landing_to_bronze` | `5-59/15 * * * *` | 5 min |
| `bronze_to_silver` | `10-59/15 * * * *` | 10 min |
| `silver_to_gold` | `15-59/15 * * * *` | 15 min |

Isso significa que, a cada 15 minutos, o pipeline completo é executado em sequência.

---

## Passo 8: Validar os Resultados

### 8.1 Verificar manifestos de execução

Os manifestos JSON documentam cada execução. Baixe e inspecione:

```bash
# Instalar AWS CLI (ou usar mc do MinIO)
pip install awscli

# Configurar AWS CLI para MinIO
aws configure --profile minio
# AWS Access Key ID: minioadmin
# AWS Secret Access Key: minioadmin
# Default region: us-east-1

# Listar manifestos
aws --profile minio --endpoint-url http://localhost:9000 s3 ls s3://datalake/landing/_control/
aws --profile minio --endpoint-url http://localhost:9000 s3 ls s3://datalake/bronze/_control/
aws --profile minio --endpoint-url http://localhost:9000 s3 ls s3://datalake/silver/_control/
aws --profile minio --endpoint-url http://localhost:9000 s3 ls s3://datalake/gold/_control/

# Baixar e visualizar um manifesto
aws --profile minio --endpoint-url http://localhost:9000 s3 cp \
  s3://datalake/gold/_control/silver_to_gold/.../manifest.json \
  /tmp/manifest_gold.json
cat /tmp/manifest_gold.json | python -m json.tool
```

### 8.2 Executar testes unitários

```bash
PYTHONPYCACHEPREFIX=/tmp/engenharia_dados_pycache \
  python3 -m unittest discover -s tests -v
```

Saída esperada (parcial):

```text
test_default_collections_match_source_model (tests.test_mongodb_landing.MongoDBLandingHelpersTest) ... ok
test_checkpoint_round_trip_is_utc (tests.test_mongodb_landing.MongoDBLandingHelpersTest) ... ok
...
test_scd2_dimensions_declare_keys_and_static_calendar (tests.test_silver_gold.SilverGoldHelpersTest) ... ok
test_manifest_aggregates_scd2_version_totals (tests.test_silver_gold.SilverGoldHelpersTest) ... ok

----------------------------------------------------------------------
Ran 30+ tests in X.XXXs

OK
```

### 8.3 Inspecionar tabelas Delta (PySpark local)

```bash
pip install ".[spark]"
```

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("validacao") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

# Ler dimensao Gold
df = spark.read.format("delta").load("s3a://datalake/gold/ecommerce/dim_cliente")
df.show(5)
df.count()

# Ler fato Gold
df_vendas = spark.read.format("delta").load("s3a://datalake/gold/ecommerce/fato_vendas")
df_vendas.show(5)
df_vendas.groupBy("ano").count().orderBy("ano").show()

# Verificar quality log
df_quality = spark.read.format("delta").load("s3a://datalake/silver/_control/quality_log")
df_quality.show(5)
```

---

## Troubleshooting

### Problema: `Connection refused` ao acessar MongoDB

**Causa**: MongoDB ainda não está pronto ou `.env` incorreto.

**Solução**:
```bash
docker compose ps mongodb
# Aguarde STATUS=healthy

# Verificar logs
docker compose logs mongodb --tail 50
```

### Problema: `NoSuchBucket` ao criar estruturas

**Causa**: MinIO não está acessível ou `S3_ENDPOINT_URL` incorreto no `.env`.

**Solução**:
```bash
docker compose ps minio
# Verificar se MinIO está healthy

# Verificar .env
grep S3_ENDPOINT_URL .env
# Deve ser: S3_ENDPOINT_URL=http://localhost:9000
```

### Problema: DAGs não aparecem na UI do Airflow

**Causa**: `airflow-dag-processor` ainda não processou os arquivos ou erro de importação.

**Solução**:
```bash
# Verificar erros de importação
docker compose exec airflow-apiserver airflow dags list-import-errors

# Verificar logs do dag-processor
docker compose logs airflow-dag-processor --tail 100

# Verificar se os arquivos estão no volume montado
docker compose exec airflow-apiserver ls -la /opt/airflow/dags/
```

### Problema: Spark job falha com `ClassNotFoundException`

**Causa**: Pacotes Delta Lake ou Hadoop AWS não foram carregados.

**Solução**: Verificar se `SPARK_PACKAGES` está configurado no `.env` e se a imagem Docker foi buildada corretamente.

```bash
# Rebuildar a imagem Airflow
docker compose build --no-cache airflow-apiserver
docker compose up -d airflow-apiserver
```

### Problema: `ModuleNotFoundError` nos scripts Python

**Causa**: Ambiente virtual não ativado ou dependências não instaladas.

**Solução**:
```bash
source .venv/bin/activate
pip install -e ".[dataset,infra,spark]"
```

### Problema: Porta `8080` já em uso

**Causa**: Outro serviço está usando a porta 8080.

**Solução**: Modifique o `docker-compose.yml` para mapear outra porta (ex: `8081:8080`).

```yaml
ports:
  - "8081:8080"
```

---

## Comandos Úteis para Operação

### Verificar logs de um serviço

```bash
docker compose logs airflow-scheduler --tail 100 -f
```

### Parar todos os serviços (mantém dados)

```bash
docker compose down
```

### Parar e remover volumes (apaga dados)

```bash
docker compose down -v
```

### Resetar o Airflow (migrações e dados)

```bash
docker compose down
docker volume rm engenharia_dados_final_airflow_postgres_data
docker compose up -d airflow-init
docker compose up -d airflow-apiserver airflow-scheduler airflow-dag-processor airflow-triggerer
```

### Executar um teste específico

```bash
python -m unittest tests.test_mongodb_landing -v
```

### Gerar apenas os CSVs (sem carregar no MongoDB)

```bash
python dataset/scripts_py/gerar_dados.py
```
